<div class="knee-title">
<h1>&#129517; Raptor CoAtNet, three training snapshots blended</h1>
<p>Three fine-tunes of the same public, CC0-licensed CoAtNet-384 checkpoint, each trained on a differently-preprocessed corpus, rank-blended together score 0.9288 on the 58 gold-labeled studies - up from 0.9255 for the strongest one alone. Getting the base recipe right took fixing five real bugs in an earlier from-scratch replication; getting the ensemble right took finding the exact per-checkpoint preprocessing config each snapshot was actually trained on, not assuming they all share one recipe.</p>
</div>

In [ ]:
from IPython.display import HTML
HTML("""
<style>
.knee-title{background:linear-gradient(90deg,#1C6E8C 0%,#2E8B57 100%);color:#ffffff;
  font-family:'Segoe UI',sans-serif;padding:16px 22px;border-radius:12px;margin:6px 0 10px 0;}
.knee-title h1{margin:0;font-size:22px;font-weight:700;}
.knee-title p{margin:6px 0 0 0;font-size:13px;opacity:0.92;}
.section{color:#1C6E8C;font-family:'Segoe UI',sans-serif;font-size:15px;font-weight:700;
  margin:10px 0 2px 0;padding-bottom:3px;border-bottom:2px solid #DCEEF0;}
.note{background:#EAF4F4;border-left:5px solid #1C6E8C;padding:9px 15px;border-radius:0 8px 8px 0;
  color:#22333B;font-family:'Segoe UI',sans-serif;font-size:13.5px;line-height:1.55;margin:4px 0 10px 0;}
.note.good{background:#EAF7EF;border-left-color:#2E8B57;}
.note.warn{background:#FFF6E9;border-left-color:#F2A541;}
.note code{background:#ffffffa8;padding:1px 5px;border-radius:4px;}
</style>
""")

In [ ]:
PALETTE = {"teal": "#1C6E8C", "green": "#2E8B57", "amber": "#F2A541", "rose": "#C1443C", "ink": "#22333B"}
import matplotlib.pyplot as plt
plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white", "axes.edgecolor": "#B9C7C9",
    "axes.labelcolor": PALETTE["ink"], "axes.titlecolor": PALETTE["teal"], "axes.titleweight": "bold",
    "xtick.color": PALETTE["ink"], "ytick.color": PALETTE["ink"], "font.family": "sans-serif",
    "axes.grid": True, "grid.color": "#E7EEEF", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
})

<div class="section">&#128218; What changed, twice</div>
<div class="note warn">An earlier version of this notebook reconstructed the checkpoint's architecture from its state dict alone and reached 0.8527 on the gold set - a full 0.07 below the 0.9214 the checkpoint's own metadata claims. Pulling the actual public source behind the checkpoint (<code>dreaddevelopment/knee-mri-twelve-findings-from-a-single-model</code>, a 0.924-public-LB notebook) turned up five concrete bugs: wrong normalization (buffers guessed inside the model instead of the real external ImageNet mean/std), a wrong attention-head activation and an unnecessary presence-masking path, missing DICOM modality LUT and MONOCHROME1 handling, percentile normalization computed per slice instead of per slot, and a 15-window sample instead of the real 42 drawn from a 64-slice volume. Fixing all five and reloading with <code>strict=True</code> brought the gold-only score to 0.9213, then a window-count sweep found a 56-60 plateau above the shipped 42, landing on 0.9255.</div>
<div class="note good">This version adds a second real gain. The checkpoint author published seven fine-tunes of the identical CoAtNet architecture, one per differently-preprocessed corpus (named <code>widedense</code>, <code>maxspan</code> - the one used above, <code>widefov</code>, <code>fullspan</code>, <code>native384</code>, <code>finespacing</code>, <code>native384dense</code>). An earlier attempt at blending four of these ("arms") failed - solo AUCs came back near random - because every arm was fed through the <code>maxspan</code> recipe regardless of what it was actually trained on. Reading a competitor's public notebook that blends these same checkpoints turned up their exact per-checkpoint build resolution, slot layout and depth-span settings. Rebuilding two arms (<code>native384</code> and <code>native384dense</code>) with their own correct settings brought both to genuinely strong solo scores (0.9116 and 0.9170), and a rank-blend of all three reaches <b>0.9288</b>.</div>

<div class="section">&#9881;&#65039; Setup</div>

In [ ]:
import os, sys, time, random, hashlib, warnings, gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import pydicom
import cv2
import timm
from pydicom.pixel_data_handlers.util import apply_modality_lut

from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| gpu count:", torch.cuda.device_count())

<div class="section">&#128193; Data</div>
<div class="note">No labels are needed here - every model below is a finished, already-trained checkpoint, so this notebook only loads the competition's study/series metadata to know which DICOM series belong to which study.</div>

In [ ]:
ROOT = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_TARGETS = len(TARGETS)

train = pd.read_csv(ROOT / "train.csv")
test = pd.read_csv(ROOT / "test.csv")
train_series = pd.read_csv(ROOT / "train_series.csv")
test_series = pd.read_csv(ROOT / "test_series.csv")
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")
gold_mask = train[TARGETS].notna().any(axis=1)
gold_idx = np.where(gold_mask.values)[0]
gold_ids = train.loc[gold_mask, "StudyInstanceUID"].tolist()
gold_y_true = train[TARGETS].values[gold_idx]
train_ids = train.StudyInstanceUID.tolist()
test_ids = test.StudyInstanceUID.tolist()
train_ser_records = {k: v.to_dict("records") for k, v in train_series.groupby("StudyInstanceUID")}
test_ser_records = {k: v.to_dict("records") for k, v in test_series.groupby("StudyInstanceUID")}
print("train:", train.shape, "| test:", test.shape, "| gold-labeled:", int(gold_mask.sum()))

<div class="section">&#129517; Shared model class and DICOM preprocessing</div>
<div class="note good">Every arm below is the same architecture - a CoAtNet-384 backbone with a per-finding attention-pooling head - so one class and one set of DICOM-reading functions serve all three. <code>strict=True</code> on each checkpoint load only succeeds on an exact architecture match; it isn't a formality, it's the actual proof each checkpoint fits this class.</div>

In [ ]:
def build_backbone(arch, pretrained=False):
    # CoAtNet/MaxViT are conv-attention hybrids: no CLS token, no interpolatable pos-embed,
    # so they need average pooling rather than the token pooling a plain ViT would use.
    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))
    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))
    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)
    if is_vit:
        kw.update(global_pool="token", dynamic_img_size=True)
    else:
        kw.update(global_pool="avg")
    return timm.create_model(arch, **kw)


class RaptorClassifier(nn.Module):
    """Per-finding attention pooling over the study's windows - each of the twelve findings
    gets its own attention weights, so a cruciate tear visible on two sagittal slices and
    osteoarthritis spread across many coronal ones don't have to share one pooled score."""
    def __init__(self, backbone, F_dim, n=12, drop=0.2):
        super().__init__()
        self.backbone = backbone
        self.norm = nn.LayerNorm(F_dim)
        self.att = nn.Sequential(nn.Linear(F_dim, 256), nn.Tanh(), nn.Dropout(drop), nn.Linear(256, n))
        self.clsW = nn.Parameter(torch.zeros(n, F_dim))
        self.clsb = nn.Parameter(torch.zeros(n))
        nn.init.trunc_normal_(self.clsW, std=0.02)
        self.n = n

    def encode(self, x):
        B, K = x.shape[:2]
        f = self.backbone(x.flatten(0, 1))
        return f.view(B, K, -1)

    def head(self, feats):
        h = self.norm(feats)
        a = torch.softmax(self.att(h), dim=1)
        pooled = torch.einsum("bkn,bkf->bnf", a, h)
        return (pooled * self.clsW).sum(-1) + self.clsb

    def forward(self, x):
        return self.head(self.encode(x))


def load_arm(ckpt_path):
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    bb = build_backbone(ck.get("arch", "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k"), pretrained=False)
    model = RaptorClassifier(bb, F_dim=bb.num_features)
    model.load_state_dict(ck["model"], strict=True)
    model.eval().to(DEVICE)
    print("loaded", ckpt_path, "| strict load OK | res:", ck.get("res"), "| its own claimed gold_auc:", ck.get("gold_auc"))
    return model, int(ck.get("res", 384))

<div class="section">&#129513; Building a fixed input, per checkpoint's own recipe</div>
<div class="note">No two studies look alike - the number of series and slices per series both vary - so every recipe here fills a fixed-size volume in a fixed slot order, preferring a different series per slot when more than one is available, leaving a missing slot at zero and telling the model to skip it. What differs <i>between checkpoints</i> is the exact shape of that fixed volume: the main <code>maxspan</code> checkpoint wants a 64-slice volume across five slots (18/14/12/8/12) built at 336px; <code>native384</code> wants a smaller 44-slice volume (12/10/8/6/8) built natively at 384px; <code>native384dense</code> keeps the 64-slice layout but also builds natively at 384px. Every slice is still cropped to a 140&nbsp;mm box from the DICOM pixel spacing, modality-LUT corrected, and MONOCHROME1-inverted where needed, and every slot's slices share one percentile normalization rather than being rescaled independently.</div>

In [ ]:
CROP_MM = 140.0
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Slot layouts: (plane, fluid-sensitive flag, slice count). -1 fluid means "no preference".
SLOTS_64 = [("Sagittal", 1, 18), ("Sagittal", 0, 14), ("Coronal", 1, 12),
            ("Coronal", 0, 8), ("Axial", -1, 12)]
SLOTS_44 = [("Sagittal", 1, 12), ("Sagittal", 0, 10), ("Coronal", 1, 8),
            ("Coronal", 0, 6), ("Axial", -1, 8)]


def _order_and_meta(series_dir):
    fs = sorted(os.listdir(series_dir))
    recs, ps_list = [], []
    for fn in fs:
        f = os.path.join(series_dir, fn)
        try:
            h = pydicom.dcmread(f, stop_before_pixels=True)
            iop = getattr(h, "ImageOrientationPatient", None)
            ipp = getattr(h, "ImagePositionPatient", None)
            if iop is not None and ipp is not None and len(iop) == 6:
                r = np.array(iop[:3], float); c = np.array(iop[3:], float)
                n = np.cross(r, c); pos = float(np.dot(np.array(ipp, float), n))
            else:
                pos = float(getattr(h, "InstanceNumber", 0) or 0)
            ps = getattr(h, "PixelSpacing", None)
            ps = float(ps[0]) if ps is not None else 0.5
            ps_list.append(ps); recs.append((pos, f, ps))
        except Exception:
            recs.append((0.0, f, 0.5))
    recs.sort(key=lambda x: x[0])
    med_ps = float(np.median(ps_list)) if ps_list else 0.5
    return [(f, ps) for _, f, ps in recs], med_ps


def _read_px(f):
    d = pydicom.dcmread(f)
    a = apply_modality_lut(d.pixel_array, d).astype(np.float32)
    if str(getattr(d, "PhotometricInterpretation", "")) == "MONOCHROME1":
        a = a.max() - a
    return a


def _mm_crop_resize(a, ps, img_build):
    h, w = a.shape
    cpx = int(round(CROP_MM / max(ps, 1e-3)))
    cpx = min(cpx, min(h, w))
    y0 = (h - cpx) // 2; x0 = (w - cpx) // 2
    a = a[y0:y0 + cpx, x0:x0 + cpx]
    return cv2.resize(a, (img_build, img_build), interpolation=cv2.INTER_AREA)


def _pick_series_for_slot(rows, plane, fluid, used):
    cands = [r for r in rows if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]
    if fluid in (0, 1):
        pref = [r for r in cands if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
        if pref:
            return pref[0]
    return cands[0] if cands else None


def build_study(study_id, ser_records, split_dir, img_build, slots, span_lo, span_hi):
    maxs = sum(s[2] for s in slots)
    rows = ser_records.get(study_id, [])
    vol = np.zeros((maxs, img_build, img_build), np.uint8)
    idx = 0
    used = set()
    for plane, fluid, k in slots:
        r = _pick_series_for_slot(rows, plane, fluid, used)
        if r is None:
            idx += k
            continue
        used.add(r["SeriesInstanceUID"])
        series_dir = ROOT / split_dir / study_id / r["SeriesInstanceUID"]
        try:
            files, med_ps = _order_and_meta(series_dir)
        except FileNotFoundError:
            files, med_ps = [], 0.5
        if not files:
            idx += k
            continue
        n = len(files)
        lo, hi = int(n * span_lo), int(n * span_hi) - 1
        hi = max(hi, lo)
        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k
        arrs, pss = [], []
        for p in picks:
            fp, ps = files[min(p, n - 1)]
            try:
                arrs.append(_read_px(fp)); pss.append(ps)
            except Exception:
                arrs.append(None); pss.append(med_ps)
        valid = [a for a in arrs if a is not None]
        if valid:
            allpx = np.concatenate([a.ravel() for a in valid])
            loq, hiq = np.percentile(allpx, [2.0, 98.0])
        else:
            loq, hiq = 0.0, 1.0
        for a, ps in zip(arrs, pss):
            if idx >= maxs:
                break
            if a is None:
                idx += 1
                continue
            aw = np.clip((a - loq) / (hiq - loq + 1e-6), 0, 1)
            aw = _mm_crop_resize(aw, ps if ps > 0 else med_ps, img_build)
            vol[idx] = (aw * 255).astype(np.uint8)
            idx += 1
        if idx >= maxs:
            break
    mask = (vol.reshape(maxs, -1).sum(1) > 0).astype(np.uint8)
    return vol, mask

<div class="section">&#128269; Evaluation windows</div>
<div class="note">Three neighbouring slices stack into the three channels of one window, so the network sees a little of what lies just above and below the slice in the middle. Window centres are drawn evenly from every valid interior position the volume holds - a window can span a slot boundary and the model still gets a dense, overlapping view of the whole study.</div>

In [ ]:
def _eval_centers(mask, D, k):
    valid = np.where(mask > 0)[0]
    if len(valid) < 3:
        valid = np.arange(min(3, D))
    lo, hi = int(valid.min()), int(valid.max())
    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]
    if not cs:
        cs = [max(1, min((lo + hi) // 2, D - 2))]
    idx = np.linspace(0, len(cs) - 1, k).round().astype(int)
    return [cs[i] for i in idx]


def eval_windows(vol, mask, k, res):
    D = vol.shape[0]
    cs = _eval_centers(mask, D, k)
    wins = np.empty((len(cs), 3, res, res), np.float32)
    for j, c in enumerate(cs):
        c = max(1, min(c, D - 2))
        tri = np.stack([vol[c - 1], vol[c], vol[c + 1]], 0).astype(np.float32) / 255.0
        t = torch.from_numpy(tri)
        if t.shape[-1] != res:
            t = F.interpolate(t[None], size=(res, res), mode="bilinear", align_corners=False)[0]
        wins[j] = t.numpy()
    x = torch.from_numpy(wins)
    return (x - _MEAN) / _STD


@torch.no_grad()
def infer_probs(model, xwins, device):
    x = xwins.unsqueeze(0).to(device)
    if str(device).startswith("cuda"):
        try:
            with torch.autocast("cuda", dtype=torch.float16):
                o = torch.sigmoid(model(x).float())
            return o[0].cpu().numpy()
        except RuntimeError:
            torch.cuda.empty_cache()
    o = torch.sigmoid(model(x).float())
    return o[0].cpu().numpy()


def raptor_predict(study_ids, ser_records, split_dir, model, device, img_build, slots, span_lo, span_hi, k_eval, res):
    out = np.full((len(study_ids), N_TARGETS), 0.5, np.float32)
    for i, sid in enumerate(study_ids):
        vol, mask = build_study(sid, ser_records, split_dir, img_build, slots, span_lo, span_hi)
        xw = eval_windows(vol, mask, k=k_eval, res=res)
        out[i] = infer_probs(model, xw, device)
    return out

<div class="section">&#128193; Loading the three arms</div>
<div class="note">Requires three datasets attached: <code>dreaddevelopment/raptor-knee-maxspan</code>, <code>dreaddevelopment/raptor-knee-native384dense</code>, and <code>dreaddevelopment/raptor-knee-native384</code> (all CC0-1.0).</div>
<div class="note">Each arm's config below - build resolution, slot layout, depth span, window count - matches exactly what that checkpoint was trained on, not a shared guess. The main <code>maxspan</code> settings come from the checkpoint author's own published inference notebook; the <code>native384</code> and <code>native384dense</code> settings come from a competitor's public notebook that blends the same three checkpoints, read directly rather than reverse-engineered. An earlier attempt to blend four <i>other</i> arms without their correct per-checkpoint settings produced near-random solo predictions - the lesson held here too, just with the right settings in hand this time.</div>

In [ ]:
ARMS = [
    {"name": "maxspan-v5", "path": "/kaggle/input/datasets/dreaddevelopment/raptor-knee-maxspan/raptor_ft_coatnet_v5_full_swa.pt",
     "img_build": 336, "slots": SLOTS_64, "span": (0.06, 0.94), "k_eval": 60},
    {"name": "native384dense-v10", "path": "/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384dense/raptor_ft_coatnet_v10_full.pt",
     "img_build": 384, "slots": SLOTS_64, "span": (0.02, 0.98), "k_eval": 62},
    {"name": "native384-v8", "path": "/kaggle/input/datasets/dreaddevelopment/raptor-knee-native384/raptor_ft_coatnet_v8_full_swa.pt",
     "img_build": 384, "slots": SLOTS_44, "span": (0.06, 0.94), "k_eval": 42},
]

for arm in ARMS:
    model, res = load_arm(arm["path"])
    arm["model"] = model
    arm["res"] = res

<div class="section">&#9989; Validation</div>
<div class="note">Scored only on the 58 real gold-labeled studies, which none of these checkpoints' training ever saw. Each arm's solo score is computed first, then a rank-percentile joint weight search finds the best blend - rank blending (not raw probability averaging) is what the checkpoint author's own multi-arm code uses, and it came out slightly ahead here too.</div>

In [ ]:
def compute_masked_auc(y_true, y_pred, cols):
    aucs = {}
    for i, col in enumerate(cols):
        y, p = y_true[:, i], y_pred[:, i]
        mask = ~np.isnan(y)
        yb = (y[mask] >= 0.5).astype(int)
        if mask.sum() > 1 and len(set(yb)) > 1:
            aucs[col] = roc_auc_score(yb, p[mask])
    return aucs, (float(np.mean(list(aucs.values()))) if aucs else float("nan"))


def rankpct(x):
    order = x.argsort(0).argsort(0).astype(np.float64)
    return order / max(1, (x.shape[0] - 1))


for arm in ARMS:
    t0 = time.time()
    span_lo, span_hi = arm["span"]
    preds = raptor_predict(gold_ids, train_ser_records, "train_series", arm["model"], DEVICE,
                            arm["img_build"], arm["slots"], span_lo, span_hi, arm["k_eval"], arm["res"])
    _, auc = compute_masked_auc(gold_y_true, preds, TARGETS)
    arm["gold_preds"] = preds
    arm["gold_auc"] = auc
    print(f"{arm['name']:20s} solo gold-only AUC: {auc:.4f}  ({time.time()-t0:.0f}s)")

best_auc, best_w = -1.0, None
step = 0.05
grid = np.arange(0.0, 1.0 + 1e-9, step)
ranked = [rankpct(arm["gold_preds"]) for arm in ARMS]
for w0 in grid:
    for w1 in grid:
        if w0 + w1 > 1.0 + 1e-9:
            continue
        w2 = 1.0 - w0 - w1
        if w2 < -1e-9:
            continue
        w2 = max(w2, 0.0)
        ens = w0 * ranked[0] + w1 * ranked[1] + w2 * ranked[2]
        _, auc = compute_masked_auc(gold_y_true, ens, TARGETS)
        if auc > best_auc:
            best_auc, best_w = auc, (w0, w1, w2)

print(f"\nbest blend: {ARMS[0]['name']}={best_w[0]:.2f} {ARMS[1]['name']}={best_w[1]:.2f} {ARMS[2]['name']}={best_w[2]:.2f} -> {best_auc:.4f}")
print(f"delta over strongest solo arm: {best_auc - max(a['gold_auc'] for a in ARMS):+.4f}")

<div class="section">&#128203; Submission</div>

In [ ]:
for arm in ARMS:
    t0 = time.time()
    span_lo, span_hi = arm["span"]
    arm["test_preds"] = raptor_predict(test_ids, test_ser_records, "test_series", arm["model"], DEVICE,
                                        arm["img_build"], arm["slots"], span_lo, span_hi, arm["k_eval"], arm["res"])
    print(f"{arm['name']:20s} test predict elapsed: {time.time()-t0:.0f}s")

ranked_test = [rankpct(arm["test_preds"]) for arm in ARMS]
pred_test = best_w[0] * ranked_test[0] + best_w[1] * ranked_test[1] + best_w[2] * ranked_test[2]

pred_df = pd.DataFrame(pred_test, columns=TARGETS)
pred_df.insert(0, "StudyInstanceUID", test_ids)

submission = sample_sub[["StudyInstanceUID"]].merge(pred_df, on="StudyInstanceUID", how="left")
submission[TARGETS] = submission[TARGETS].fillna(0.5)
submission = submission[sample_sub.columns.tolist()]

assert list(submission.columns) == list(sample_sub.columns)
assert len(submission) == len(sample_sub)
assert set(submission.StudyInstanceUID) == set(sample_sub.StudyInstanceUID)
assert submission[TARGETS].isna().sum().sum() == 0
assert ((submission[TARGETS] >= 0) & (submission[TARGETS] <= 1)).all().all()

submission.to_csv("submission.csv", index=False)
print("submission.csv written:", submission.shape)
submission

<div class="section">&#128204; Notes</div>
<div class="note good">Solo gold-only AUC: <code>maxspan-v5</code> 0.9255 (the same checkpoint the previous version of this notebook shipped, scoring 0.928 on the real leaderboard at its 42-window setting), <code>native384dense-v10</code> 0.9170, <code>native384-v8</code> 0.9116. Rank-blended at the weights found above: <b>0.9288</b>. This is the first ensembling gain of any real size found in this project since the base recipe was fixed - every other attempt at adding a second model added at most a few thousandths of an AUC point, because none of those secondary models were anywhere near raptor's own accuracy. These three checkpoints all share raptor's own architecture and general training recipe, just fit on differently-preprocessed corpora - close enough in strength that combining them adds real information instead of diluting it.</div>
<div class="note">Every other real alternative model found or built was tested properly and none added more than a few thousandths of an AUC point once blended: three more checkpoints in the same architecture family with recipes we could verify (<code>widefov</code> v6 0.9108, <code>fullspan</code> v7 0.9063, <code>finespacing</code> v9 0.8985 solo - all decent, all driven to zero weight by the joint search); the competitor's own real out-of-fold predictions for a 5-fold DINOv2 model (0.8400 solo, +0.0000 blended) and three RadImageNet-ResNet50 variants (0.83-0.86 solo, +0.0006 blended, best case); and a properly task-fine-tuned OrthoFoundation-L (a DINOv3 model pretrained on ~1.25M knee images), taken from a 0.75 frozen-embedding probe to a real 0.8124 solo AUC via three epochs of attention-pooling fine-tuning - still only +0.0004 blended. Four checkpoints in the raptor family remain untested (<code>widedense</code> v4/v4-swa and two others) for lack of a verified per-checkpoint recipe - not attempted, to avoid repeating the near-random-prediction failure that a wrong config produces.</div>
<div class="note warn">All of the above is validated against 58 gold-labeled studies - informative, but a small enough sample that the real test is the leaderboard once this gets submitted. Two competing public notebooks scoring higher than this (0.936, 0.937) were read in full: their edge comes mainly from blending a second, independently strong DINOv2 model (not a trick this notebook could just copy in), and their own code shows real signs of small-sample overfitting on their internal validation set - a caution worth applying to the weights found in this notebook too. Fresh discussion-thread research points at continued self-supervised pretraining before fine-tuning as the more likely path past this ceiling, not more models to blend in.</div>